# Branch propagation

This notebook is an interactive monitor for `branch_propagation.py`, which is the sole implementation of the runnable continuation workflow. An attempt may increase resolution or the epoch budget without accepting a point; inspect `accepted_point` in its summary.

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt

from branch_propagation import branch_summary, load_branch, propagate_branch

## Load and inspect resumable state

Set `perpendicularize=True` when time translation occupies one of the first two Hessian modes, as in the nonsymmetric bifurcation branches. Symmetry-restricted fixed-point branches normally use `False`.

In [ ]:
results_folder = 'hess_results/bif_positive_reverse'
branch = load_branch(results_folder)
branch_summary(branch)

## Optional saved-state overrides

Leave these unchanged to resume with the parameters stored in `propagation.pt`. Explicit assignments are shown so any research override remains visible in notebook history.

In [ ]:
# branch.val_epochs = 2000
# branch.epoch_limit = 100000
# branch.frequency_double_cutoff = 1e-20
# branch.epoch_double_cutoff = 1e-16
branch_summary(branch)

## Run and save continuation

`propagate_branch` saves the root `propagation.pt` after every attempt so the workflow can be resumed. The parameters below reproduce the style of the original bifurcation-propagation notebook; adjust them for other families.

In [ ]:
attempts, trainer = propagate_branch(
    branch,
    iterations=1,
    perpendicularize=True,
    parameterization='weighted',
    learning_rate=1e-20,
    step_size_min=1e-100,
    step_size_max=1.0,
    extrapolate=True,
    epoch_doubling=False,
    extrapolate_cutoff=1.5,
    print_validation=200,
    print_extrapolation=True,
)
[asdict(attempt) for attempt in attempts]

## Inspect the corrector trajectory

In [ ]:
checkpoint = trainer.checkpoints[-1]
{
    'accepted_point': attempts[-1].accepted_point,
    'initial_condition_degrees': (
        checkpoint.initial_condition.squeeze() * 180 / checkpoint.initial_condition.new_tensor(3.141592653589793)
    ).tolist(),
    'period': checkpoint.T.item(),
    'train_loss': checkpoint.train_loss.item(),
    'validation_loss': checkpoint.val_loss,
}

In [ ]:
epochs = trainer.get_checkpoints_data('epoch')
train_losses = [value.item() for value in trainer.get_checkpoints_data('train_loss')]
validation_losses = trainer.get_checkpoints_data('val_loss')

fig, ax = plt.subplots()
ax.plot(epochs, train_losses, label='Variational loss')
ax.plot(epochs, validation_losses, label='IVP validation loss')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend();

In [ ]:
branch_summary(branch)